# BCG X – PowerCo Customer Churn Analysis
## Step 4: Feature Engineering

**Analyst:** [Your Name]  
**Date:** 2025  

**Objective:** Engineer meaningful features from the clean dataset to improve churn prediction model performance.

We follow the **four-question framework**:
1. Can we **remove** any columns?
2. Can we **expand** date columns into useful features?
3. Can we **combine** columns to create better features?
4. Can we **merge** datasets if applicable?

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully!')

## 2. Load the Cleaned Dataset

In [ ]:
df = pd.read_csv('clean_data_after_eda.csv')

print(f'Dataset shape : {df.shape}')
print(f'Churn rate    : {df["churn"].mean()*100:.2f}%')
print()
print('Column list:')
for col in df.columns:
    print(f'  {col}: {df[col].dtype}')

In [ ]:
# Keep a copy of the original for reference
df_original = df.copy()
print('Original dataset backed up.')

---
## QUESTION 1: Can we remove any columns?

Columns to remove:
- `id` — unique customer identifier, not a predictive feature

> We keep all other columns for now and evaluate their importance during modeling.

In [ ]:
# Check unique values to find low-variance or ID-like columns
print('Unique value counts per column:')
for col in df.columns:
    n_unique = df[col].nunique()
    print(f'  {col}: {n_unique} unique values')

In [ ]:
# Drop non-predictive columns
cols_to_drop = ['id']
df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped columns: {cols_to_drop}')
print(f'Shape after drop: {df.shape}')

---
## QUESTION 2: Can we expand date columns into useful features?

Date columns in raw form (`date_activ`, `date_end`, `date_modif_prod`, `date_renewal`) are not directly useful for ML models.

We will extract:
- **tenure_days** — how long the customer has been with PowerCo
- **days_to_end** — days until contract ends (urgency of churn?)
- **months_to_renewal** — months until next renewal
- **months_since_modif** — months since last product modification
- **activ_month / activ_year** — seasonality patterns

In [ ]:
# Parse date columns
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Date columns parsed:')
for col in date_cols:
    print(f'  {col}: {df[col].dtype}, nulls={df[col].isnull().sum()}')

In [ ]:
# Reference date: approximately when the dataset was collected
ref_date = pd.Timestamp('2016-01-01')

# Feature: how long customer has been active (tenure)
df['tenure_days'] = (ref_date - df['date_activ']).dt.days

# Feature: how many days until the contract ends
df['days_to_end'] = (df['date_end'] - ref_date).dt.days

# Feature: months until next renewal
df['months_to_renewal'] = (df['date_renewal'] - ref_date).dt.days / 30

# Feature: months since last product modification
df['months_since_modif'] = (ref_date - df['date_modif_prod']).dt.days / 30

# Feature: activation month and year (seasonality)
df['activ_month'] = df['date_activ'].dt.month
df['activ_year']  = df['date_activ'].dt.year

# Drop raw date columns (no longer needed)
df.drop(columns=date_cols, inplace=True)

print('New date-derived features created:')
new_date_feats = ['tenure_days', 'days_to_end', 'months_to_renewal', 'months_since_modif', 'activ_month', 'activ_year']
print(df[new_date_feats].describe().T[['mean','min','max']].round(2))

In [ ]:
# Visualize: Tenure vs Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, group in df.groupby('churn')['tenure_days']:
    axes[0].hist(group.dropna(), bins=40, alpha=0.6,
                 label='Churned' if label == 1 else 'Not Churned')
axes[0].set_title('Customer Tenure by Churn Status', fontweight='bold')
axes[0].set_xlabel('Tenure (days)')
axes[0].set_ylabel('Count')
axes[0].legend()

for label, group in df.groupby('churn')['days_to_end']:
    axes[1].hist(group.dropna(), bins=40, alpha=0.6,
                 label='Churned' if label == 1 else 'Not Churned')
axes[1].set_title('Days to Contract End by Churn Status', fontweight='bold')
axes[1].set_xlabel('Days to End')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Date-Derived Features vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('date_features_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

---
## QUESTION 3: Can we combine columns to create better features?

### 3A. Price Sensitivity Features
Key hypothesis: customers churn because of price sensitivity → create features that capture **price differences and changes**.

In [ ]:
# Price difference: off-peak vs peak (bigger diff = more incentive to switch)
df['price_off_vs_peak_diff']  = df['var_year_price_off_peak'] - df['var_year_price_peak']

# Price sensitivity ratio: how much peak price relative to off-peak
df['price_sensitivity_ratio'] = df['var_year_price_peak'] / (df['var_year_price_off_peak'] + 1e-9)

# Price change over time: 6-month vs 1-year off-peak change
df['price_change_6m_vs_1y']   = df['var_6m_price_off_peak'] - df['var_year_price_off_peak']

# Average price change over the year (across all tariff periods)
df['avg_price_change_yearly'] = (
    df['var_year_price_off_peak_var'] +
    df['var_year_price_peak_var'] +
    df['var_year_price_mid_peak_var']
) / 3

# Average price change over 6 months
df['avg_price_change_6m'] = (
    df['var_6m_price_off_peak_var'] +
    df['var_6m_price_peak_var'] +
    df['var_6m_price_mid_peak_var']
) / 3

print('Price sensitivity features created:')
price_feats = ['price_off_vs_peak_diff','price_sensitivity_ratio',
               'price_change_6m_vs_1y','avg_price_change_yearly','avg_price_change_6m']
print(df[price_feats].describe().T[['mean','std','min','max']].round(4))

In [ ]:
# Visualize: Price sensitivity ratio by churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='avg_price_change_yearly', by='churn', ax=axes[0])
axes[0].set_title('Avg Yearly Price Change by Churn', fontweight='bold')
axes[0].set_xlabel('Churn (0=No, 1=Yes)')
axes[0].set_ylabel('Avg Price Change (Yearly)')

df.boxplot(column='avg_price_change_6m', by='churn', ax=axes[1])
axes[1].set_title('Avg 6-Month Price Change by Churn', fontweight='bold')
axes[1].set_xlabel('Churn (0=No, 1=Yes)')
axes[1].set_ylabel('Avg Price Change (6m)')

plt.suptitle('')
plt.tight_layout()
plt.savefig('price_features_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

### 3B. Consumption Features

In [ ]:
# Ratio of last month consumption vs 12-month total (sudden drop = risk of churn?)
df['cons_ratio_last_to_12m'] = df['cons_last_month'] / (df['cons_12m'] + 1e-9)

# Gap between forecast and actual consumption
df['forecast_vs_actual'] = df['forecast_cons_12m'] - df['cons_12m']

# Whether customer uses gas at all
df['has_gas_usage'] = (df['cons_gas_12m'] > 0).astype(int)

print('Consumption features created:')
cons_feats = ['cons_ratio_last_to_12m','forecast_vs_actual','has_gas_usage']
print(df[cons_feats].describe().T[['mean','std']].round(4))

### 3C. Margin / Revenue Features

In [ ]:
# Net vs gross margin ratio (profitability quality)
df['margin_ratio'] = df['margin_net_pow_ele'] / (df['margin_gross_pow_ele'] + 1e-9)

# Net margin per active product
df['net_margin_per_product'] = df['net_margin'] / (df['nb_prod_act'] + 1e-9)

print('Margin features created:')
margin_feats = ['margin_ratio','net_margin_per_product']
print(df[margin_feats].describe().T[['mean','std','min','max']].round(4))

### 3D. Encode Categorical Columns

In [ ]:
# has_gas: 't'/'f' → binary
df['has_gas_bin'] = (df['has_gas'] == 't').astype(int)

# channel_sales: label encode
df['channel_sales_encoded'] = df['channel_sales'].astype('category').cat.codes

# origin_up: label encode
df['origin_up_encoded'] = df['origin_up'].astype('category').cat.codes

# Drop original categorical columns
df.drop(columns=['has_gas', 'channel_sales', 'origin_up'], inplace=True)

print('Categorical columns encoded.')
print(df[['has_gas_bin','channel_sales_encoded','origin_up_encoded']].value_counts().head(10))

---
## 4. Final Dataset Overview

In [ ]:
print('=== FEATURE ENGINEERING COMPLETE ===')
print(f'Original shape : {df_original.shape}')
print(f'Final shape    : {df.shape}')
print(f'New features   : {df.shape[1] - df_original.shape[1] + 1} columns added (net of drops)')
print()
print('Final column list:')
for col in df.columns:
    print(f'  {col}')

In [ ]:
# Null check
null_counts = df.isnull().sum()
null_cols   = null_counts[null_counts > 0]
if null_cols.empty:
    print('No missing values in final dataset.')
else:
    print('Columns with missing values:')
    print(null_cols)

In [ ]:
# Correlation with churn — top 15 features
corr_with_churn = df.corr()['churn'].drop('churn').abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
corr_with_churn.plot(kind='barh', color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.title('Top 15 Features by Correlation with Churn', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.savefig('feature_correlation_with_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 15 features correlated with churn:')
print(corr_with_churn.round(4))

## 5. Save Feature-Engineered Dataset

In [ ]:
output_path = 'feature_engineered_data.csv'
df.to_csv(output_path, index=False)
print(f'Feature-engineered dataset saved to: {output_path}')
print(f'Final shape: {df.shape}')

---
## Summary of Feature Engineering

| Category | Features Created |
|----------|------------------|
| **Removed** | `id` (non-predictive) |
| **Date-Derived** | `tenure_days`, `days_to_end`, `months_to_renewal`, `months_since_modif`, `activ_month`, `activ_year` |
| **Price Sensitivity** | `price_off_vs_peak_diff`, `price_sensitivity_ratio`, `price_change_6m_vs_1y`, `avg_price_change_yearly`, `avg_price_change_6m` |
| **Consumption** | `cons_ratio_last_to_12m`, `forecast_vs_actual`, `has_gas_usage` |
| **Margin** | `margin_ratio`, `net_margin_per_product` |
| **Encoded Categoricals** | `has_gas_bin`, `channel_sales_encoded`, `origin_up_encoded` |

**Total new features added: 19**

**Next Step → Step 5: Modeling and Evaluation** (Random Forest / Logistic Regression for churn prediction)